In [0]:
# %sql
# CREATE CATALOG IF NOT EXISTS workspace;
# CREATE SCHEMA IF NOT EXISTS workspace.bronze;

In [0]:
# Databricks notebook source
import pandas as pd
import requests
from io import BytesIO
from datetime import datetime, timezone
import pytz

# Configurações de exibição
pd.set_option('display.max_columns', None)

In [0]:
MAPEAMENTO_COLUNAS = {
    'regiao sigla': 'regiao_sigla',
    'regiao - sigla': 'regiao_sigla',
    'estado sigla': 'estado_sigla',
    'estado - sigla': 'estado_sigla',
    'municipio': 'municipio',
    'município': 'municipio',
    'revenda': 'revenda',
    'cnpj da revenda': 'cnpj_revenda',
    'nome da rua': 'nome_rua',
    'numero rua': 'numero_rua',
    'complemento': 'complemento',
    'bairro': 'bairro',
    'cep': 'cep',
    'produto': 'produto',
    'data da coleta': 'data_coleta',
    'valor de venda': 'preco_medio_revenda',
    'preco medio revenda': 'preco_medio_revenda',
    'preço médio revenda': 'preco_medio_revenda',
    'valor de compra': 'valor_compra',
    'unidade de medida': 'unidade_medida',
    'bandeira': 'bandeira',
    'data inicio': 'data_inicio',
    'data final': 'data_final',
    'preco minimo revenda': 'preco_minimo_revenda',
    'preco maximo revenda': 'preco_maximo_revenda',
    'margem media revenda': 'margem_media_revenda',
    'desvio padrao revenda': 'desvio_padrao_revenda',
    'coeficiente de variacao revenda': 'coeficiente_variacao_revenda',
}

def limpar_nome_original(nome):
    """Remove BOM e caracteres estranhos, normaliza espaços e hífens."""
    nome = nome.strip().replace('\ufeff', '')
    nome = ' '.join(nome.split())
    return nome.lower()

In [0]:
# URLs dos CSVs da ANP (últimas 4 semanas)
URLS = {
    "diesel_gnv": "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/qus/ultimas-4-semanas-diesel-gnv.csv",
    "gasolina_etanol": "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/qus/ultimas-4-semanas-gasolina-etanol.csv",
    "glp": "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/qus/ultimas-4-semanas-glp.csv",
}

# Tabela Delta de destino
TABELA_DESTINO = "bronze.anp_precos_combustiveis"

# Modo de salvamento: "overwrite" (substitui) ou "append" (acumula histórico)
MODO_SALVAMENTO = "overwrite"

# Timezone para coluna de ingestão
TIMEZONE_INGESTAO = pytz.timezone("America/Sao_Paulo")  # ou None para UTC

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

In [0]:
def baixar_e_padronizar(nome_arquivo, url):
    """
    Baixa um CSV da ANP, padroniza colunas e converte tipos.
    Retorna (DataFrame, data_publicacao) onde data_publicacao é inferida do cabeçalho HTTP.
    """
    try:
        print(f"Baixando {nome_arquivo}...")
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()

        # Tenta extrair a data de última modificação do arquivo no servidor
        last_modified = resp.headers.get('Last-Modified')
        data_publicacao = None
        if last_modified:
            data_publicacao = pd.to_datetime(last_modified, utc=True)

        # Lê o CSV
        df = pd.read_csv(
            BytesIO(resp.content),
            encoding='latin1',
            sep=';',
            on_bad_lines='skip'
        )

        # Normaliza nomes e aplica mapeamento
        colunas_originais = [limpar_nome_original(col) for col in df.columns]
        colunas_padronizadas = [
            MAPEAMENTO_COLUNAS[col] if col in MAPEAMENTO_COLUNAS else col.replace(' ', '_')
            for col in colunas_originais
        ]
        df.columns = colunas_padronizadas

        # Converte colunas de data
        for col_data in ['data_coleta', 'data_inicio', 'data_final']:
            if col_data in df.columns:
                df[col_data] = pd.to_datetime(df[col_data], dayfirst=True, errors='coerce')

        # Converte colunas numéricas (vírgula como separador decimal)
        colunas_preco = [
            'preco_medio_revenda', 'preco_minimo_revenda', 'preco_maximo_revenda',
            'margem_media_revenda', 'desvio_padrao_revenda', 'coeficiente_variacao_revenda',
            'valor_compra'
        ]
        for col in colunas_preco:
            if col in df.columns:
                df[col] = df[col].astype(str).str.replace(',', '.').str.strip()
                df[col] = pd.to_numeric(df[col], errors='coerce')

        print(f"  → {len(df)} linhas carregadas.")
        return df, data_publicacao
    except Exception as e:
        print(f"  → Erro ao baixar {nome_arquivo}: {e}")
        return None, None

In [0]:
frames = []
data_publicacao_geral = None

# Baixa todos os arquivos
for nome, url in URLS.items():
    df_temp, data_pub = baixar_e_padronizar(nome, url)
    if df_temp is not None:
        frames.append(df_temp)
        if data_pub:
            if data_publicacao_geral is None or data_pub < data_publicacao_geral:
                data_publicacao_geral = data_pub

if frames:
    df_full = pd.concat(frames, ignore_index=True)
    print(f"\nTotal geral: {len(df_full)} registros.")

    # Coluna de ingestão (momento da execução)
    agora = datetime.now(timezone.utc)
    dt_ingestion = agora.astimezone(TIMEZONE_INGESTAO) if TIMEZONE_INGESTAO else agora
    df_full['dt_ingestion'] = dt_ingestion

    if data_publicacao_geral:
        df_full['data_publicacao_site'] = data_publicacao_geral
        print(f"Data de publicação inferida: {data_publicacao_geral}")
    else:
        print("Não foi possível inferir a data de publicação do site.")

    # Amostra para conferência
    if 'preco_medio_revenda' in df_full.columns:
        print("\nAmostra de preços médios de revenda:")
        display(df_full[['estado_sigla', 'municipio', 'produto', 'preco_medio_revenda']].dropna().head(10))

    # Converte para Spark DataFrame
    spark_df = spark.createDataFrame(df_full)

    # Salva como tabela Delta
    spark_df.write \
        .mode(MODO_SALVAMENTO) \
        .option("mergeSchema", "true") \
        .format("delta") \
        .saveAsTable(TABELA_DESTINO)

    print(f"\nTabela '{TABELA_DESTINO}' salva com sucesso! (modo: {MODO_SALVAMENTO})")
    print(f"Registros processados: {spark_df.count()}")
    display(spark_df.limit(5))
else:
    print("Nenhum dado foi baixado.")